# RFS-FIM Fetch (DRAFT) — draw AOI on a map

Same pipeline as before, but **Step 1 lets you draw the AOI on an interactive map**
instead of reading a shapefile. The drawn shape is converted to its bounding box,
and the rest of the workflow is unchanged.

Read-only on S3; all outputs are written locally to `Results/`.

## Setup

In [65]:
# run once if needed
# %pip install s3fs geopandas rioxarray rasterio ipyleaflet ipywidgets

import math, os, re
import numpy as np
import geopandas as gpd
import s3fs
from shapely.geometry import box, shape

fs = s3fs.S3FileSystem(anon=True)
BUCKET = "floodmap-sandbox"
DEM = "fabdem"
TIF_NAME = "flows_2,5,10,25,50,100.tif"
RETURN_PERIODS = [2, 5, 10, 25, 50, 100]

## 1. Accept AOI — draw it on the map

Run the cell below. Use the **polygon** or **rectangle** tool on the left edge of the
map to draw your area of interest. Draw as many shapes as you like — the **last** one
is the one we use. The map starts over the bucket's data region (Cuba) on satellite
imagery.

*Note:* the map is an `ipyleaflet` widget. If it shows up blank in VS Code, the Jupyter
widget support isn't active — install `ipywidgets`/`ipyleaflet` and **restart the
kernel** (see commented `%pip` line in Setup).

In [66]:
from ipyleaflet import Map, DrawControl, basemaps

m = Map(center=(22.10, -78.60), zoom=10,
        basemap=basemaps.Esri.WorldImagery, scroll_wheel_zoom=True)
m.layout.height = "800px"   # double the default 400px

draw = DrawControl(
    polygon={"shapeOptions": {"color": "#e6550d", "fillOpacity": 0.2}},
    rectangle={"shapeOptions": {"color": "#e6550d", "fillOpacity": 0.2}},
    circle={}, circlemarker={}, polyline={}, marker={},
)

drawn = {"geojson": None}
def _on_draw(control, action, geo_json):
    drawn["geojson"] = geo_json
    print(f"Captured a {geo_json['geometry']['type']} ({action}).")
draw.on_draw(_on_draw)

m.add(draw)
m

Map(center=[22.1, -78.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### Capture the drawing → bbox

Converts the last drawn shape into its axis-aligned bounding box (a rectangle that
fully encompasses what you drew). Set `AOI_NAME` to label the output files.

In [68]:
AOI_NAME = "drawn_aoi"      # rename to label outputs, e.g. "Moron"

if drawn["geojson"] is None:
    raise RuntimeError("No shape captured — draw a polygon/rectangle on the map above, then re-run.")

poly = shape(drawn["geojson"]["geometry"])          # ipyleaflet draws in EPSG:4326 (lon/lat)
minx, miny, maxx, maxy = poly.bounds                # tight envelope -> always encloses the shape
aoi = gpd.GeoDataFrame({"name": [AOI_NAME]},
                       geometry=[box(minx, miny, maxx, maxy)], crs="EPSG:4326")
aoi_name = AOI_NAME
aoi_bounds = aoi.total_bounds

print(f"AOI '{aoi_name}' bbox (min_lon, min_lat, max_lon, max_lat):")
print(aoi_bounds)

AOI 'drawn_aoi' bbox (min_lon, min_lat, max_lon, max_lat):
[-78.664856  22.069414 -78.587608  22.136214]


## 2. Accept return-period selection

Choose any subset of the available periods (one GeoTIFF will be produced per period).

In [75]:
SELECTED_RPS = [2, 5, 10, 25, 50, 100]            # or list(RETURN_PERIODS) for all
assert all(rp in RETURN_PERIODS for rp in SELECTED_RPS), f"Pick from {RETURN_PERIODS}"
print("Selected return periods:", SELECTED_RPS)

Selected return periods: [2, 5, 10, 25, 50, 100]


## 3. Derive the tile range

Floor the minimums and ceil the maximums of the bbox to integer tile boundaries.

In [76]:
min_lon = math.floor(aoi_bounds[0])
min_lat = math.floor(aoi_bounds[1])
max_lon = math.ceil(aoi_bounds[2])
max_lat = math.ceil(aoi_bounds[3])
print(f"lon: {min_lon} -> {max_lon}   lat: {min_lat} -> {max_lat}")

lon: -79 -> -78   lat: 22 -> 23


## 4. Build candidate tile URLs and keep the ones that exist

In [77]:
urls = []
for lon_to_download in range(min_lon, max_lon):
    for lat_to_download in range(min_lat, max_lat):
        key = (f"{BUCKET}/tiles/lon={lon_to_download}/lat={lat_to_download}"
               f"/floodmaps/dem={DEM}/{TIF_NAME}")
        if fs.exists(key):
            urls.append(key)
            print(f"  found:   lon={lon_to_download}, lat={lat_to_download}")
        else:
            print(f"  missing: lon={lon_to_download}, lat={lat_to_download}")
print(f"\n{len(urls)} tiles to download.")

  found:   lon=-79, lat=22

1 tiles to download.


## 5. Download the tiles locally

In [78]:
os.makedirs("downloads", exist_ok=True)
local_paths = []
for url in urls:
    lon = url.split("lon=")[1].split("/")[0]
    lat = url.split("lat=")[1].split("/")[0]
    local = f"downloads/lon{lon}_lat{lat}_{TIF_NAME}"
    if not os.path.exists(local):
        fs.get(url, local)
    local_paths.append(local)
    print("  ", local)
print(f"\n{len(local_paths)} files local.")

   downloads/lon-79_lat22_flows_2,5,10,25,50,100.tif

1 files local.


## 6. Clip + mosaic → one GeoTIFF per return period (saved to `Results/`)

Pixel value is inverse to return period (brightest 100 = 2-yr core, darkest 16 =
100-yr fringe), so values pair with periods **descending** and the flood mask uses
`>=`. Dry pixels and everything outside the AOI are written as nodata (255).

In [79]:
import rioxarray
from rioxarray.merge import merge_arrays

arrays = [rioxarray.open_rasterio(p, masked=True).squeeze("band", drop=True)
          for p in local_paths]
mosaic = merge_arrays(arrays) if len(arrays) > 1 else arrays[0]

vals = sorted((int(v) for v in np.unique(mosaic.values[~np.isnan(mosaic.values)]) if v > 0),
              reverse=True)
rp_to_value = dict(zip(sorted(RETURN_PERIODS), vals))
print("return-period -> pixel-value map:", rp_to_value)

os.makedirs("Results", exist_ok=True)

written = []
for rp in SELECTED_RPS:
    thresh = rp_to_value[rp]
    mask = ((mosaic > 0) & (mosaic >= thresh)).astype("uint8")
    extent = mask.where(mask == 1, 255).astype("uint8")
    extent = extent.rio.write_crs(mosaic.rio.crs).rio.write_nodata(255)

    clipped = extent.rio.clip(aoi.geometry.values, aoi.crs, drop=True)
    out_path = os.path.join("Results", f"{aoi_name}_rp{rp}.tif")
    clipped.rio.to_raster(out_path)

    n = int((clipped == 1).sum())
    written.append(out_path)
    print(f"  wrote {out_path}  ({n:,} flooded px in AOI)")

print("\nDone. Files:", written)

return-period -> pixel-value map: {2: 100, 5: 83, 10: 66, 25: 50, 50: 33, 100: 16}
  wrote Results\drawn_aoi_rp2.tif  (8,558 flooded px in AOI)
  wrote Results\drawn_aoi_rp5.tif  (10,584 flooded px in AOI)
  wrote Results\drawn_aoi_rp10.tif  (12,094 flooded px in AOI)
  wrote Results\drawn_aoi_rp25.tif  (13,920 flooded px in AOI)
  wrote Results\drawn_aoi_rp50.tif  (15,421 flooded px in AOI)
  wrote Results\drawn_aoi_rp100.tif  (17,435 flooded px in AOI)

Done. Files: ['Results\\drawn_aoi_rp2.tif', 'Results\\drawn_aoi_rp5.tif', 'Results\\drawn_aoi_rp10.tif', 'Results\\drawn_aoi_rp25.tif', 'Results\\drawn_aoi_rp50.tif', 'Results\\drawn_aoi_rp100.tif']


### View the results on a map

Each output raster is rendered to a transparent PNG (flooded = colored, everything
else = clear) and overlaid on satellite imagery at its real-world bounds — no tile
server needed. Toggle layers with the control in the top-right. Colors go dark→light
as the return period increases (rarer events on the outside).

*(Same widget caveat as the Step 1 map: needs ipywidgets active in the kernel.)*

In [80]:
import base64
from io import BytesIO
from PIL import Image
import rasterio
from ipyleaflet import Map, ImageOverlay, LayersControl, TileLayer

# distinct color per return period (darker = more frequent / smaller event)
PALETTE = {2:(8,48,107), 5:(8,81,156), 10:(33,113,181),
           25:(66,146,198), 50:(107,174,214), 100:(189,215,231)}

def raster_to_overlay(path, rgb, name):
    with rasterio.open(path) as src:
        a = src.read(1); b = src.bounds          # b: left,bottom,right,top in lon/lat
    h, w = a.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[a == 1] = [rgb[0], rgb[1], rgb[2], 210]  # flooded -> colored; dry/nodata stay transparent
    buf = BytesIO(); Image.fromarray(rgba, "RGBA").save(buf, "PNG")
    uri = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
    return ImageOverlay(url=uri, bounds=((b.bottom, b.left), (b.top, b.right)), name=name)

cx = (aoi_bounds[0] + aoi_bounds[2]) / 2
cy = (aoi_bounds[1] + aoi_bounds[3]) / 2
result_map = Map(center=(cy, cx), zoom=12, scroll_wheel_zoom=True)
result_map.layout.height = "800px"   # double the default 400px

# Google Hybrid basemap (satellite + roads + city names baked in). lyrs=y -> hybrid
result_map.add(TileLayer(url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
                         name="Google Hybrid", attribution="Google"))

# draw rarest (largest) first so the more-frequent, smaller extents sit on top
for rp in sorted(SELECTED_RPS, reverse=True):
    path = os.path.join("Results", f"{aoi_name}_rp{rp}.tif")
    result_map.add(raster_to_overlay(path, PALETTE.get(rp, (222,45,38)), f"{rp}-yr"))

result_map.add(LayersControl(position="topright"))
result_map

Map(center=[np.float64(22.102814), np.float64(-78.626232)], controls=(ZoomControl(options=['position', 'zoom_i…